In [1]:
from google.colab import files
uploaded = files.upload()

Saving BBC News Train.csv.zip to BBC News Train.csv.zip


In [2]:
import pandas as pd

df = pd.read_csv('BBC News Train.csv.zip')
print(df.shape)
print(df.columns.tolist())
print(df['Category'].value_counts())

(1490, 3)
['ArticleId', 'Text', 'Category']
Category
sport            346
business         336
politics         274
entertainment    273
tech             261
Name: count, dtype: int64


In [3]:
print(df['Text'][0][:600])

worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness.  cynthia cooper  worldcom s ex-head of internal accounting  alerted directors to irregular accounting practices at the us telecoms giant in 2002. her warnings led to the collapse of the firm following the discovery of an $11bn (£5.7bn) accounting fraud. mr ebbers has pleaded not guilty to charges of fraud and conspiracy.  prosecution lawyers have argued that mr ebbers orchestrated a series of accounting tricks at 


In [4]:
df['word_count'] = df['Text'].str.split().str.len()
print(df['word_count'].describe())
print("Missing text:", df['Text'].isna().sum())
print("Duplicate texts:", df['Text'].duplicated().sum())

count    1490.000000
mean      385.012752
std       210.898616
min        90.000000
25%       253.000000
50%       337.000000
75%       468.750000
max      3345.000000
Name: word_count, dtype: float64
Missing text: 0
Duplicate texts: 50


In [5]:
import re

# 1. Do any duplicate texts have conflicting labels? (1 = labels agree)
dups = df[df['Text'].duplicated(keep=False)]
print("Max distinct labels per duplicated text:", dups.groupby('Text')['Category'].nunique().max())

# 2. Drop duplicates, keeping the first copy
df = df.drop_duplicates(subset='Text').reset_index(drop=True)
print("Shape after dedupe:", df.shape)

# 3. Collapse repeated spaces/newlines into single spaces
df['clean_text'] = df['Text'].apply(lambda t: re.sub(r'\s+', ' ', t).strip())
print(df['clean_text'][0][:300])

Max distinct labels per duplicated text: 1
Shape after dedupe: (1440, 4)
worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness. cynthia cooper worldcom s ex-head of internal accounting alerted directors to irregular accounting practices at the u


In [6]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(words), step):
        piece = words[start:start + chunk_size]
        chunks.append(" ".join(piece))
        if start + chunk_size >= len(words):
            break
    return chunks

rows = []
for _, r in df.iterrows():
    for i, c in enumerate(chunk_text(r['clean_text'])):
        rows.append({
            'chunk_id': f"{r['ArticleId']}_{i}",
            'ArticleId': r['ArticleId'],
            'Category': r['Category'],
            'chunk_text': c,
        })

chunks_df = pd.DataFrame(rows)
print(chunks_df.shape)
print(chunks_df['chunk_text'].str.split().str.len().describe())
print(chunks_df.iloc[0]['chunk_id'], '->', chunks_df.iloc[0]['chunk_text'][:200])

(4981, 4)
count    4981.000000
mean      132.692431
std        33.110327
min        31.000000
25%       136.000000
50%       150.000000
75%       150.000000
max       150.000000
Name: chunk_text, dtype: float64
1833_0 -> worldcom ex-boss launches defence lawyers defending former worldcom chief bernie ebbers against a battery of fraud charges have called a company whistleblower as their first witness. cynthia cooper wo


In [7]:
chunks_df.to_csv('bbc_chunks.csv', index=False)

from google.colab import files
files.download('bbc_chunks.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build the index: one vector per chunk
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=2)
tfidf_matrix = vectorizer.fit_transform(chunks_df['chunk_text'])
print("Index shape (chunks x vocabulary):", tfidf_matrix.shape)

def search_tfidf(query, k=5):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).ravel()
    top = scores.argsort()[::-1][:k]
    results = chunks_df.iloc[top].copy()
    results['score'] = scores[top]
    return results[['chunk_id', 'Category', 'score', 'chunk_text']]

hits = search_tfidf("interest rates and inflation")
for _, h in hits.iterrows():
    print(f"[{h['chunk_id']}] ({h['Category']}) score={h['score']:.3f}")
    print(h['chunk_text'][:200], "...\n")

Index shape (chunks x vocabulary): (4981, 16759)
[1088_4] (business) score=0.411
report published on wednesday the bank said that with rates at their current level inflation would rise to its 2% target within two years. however boe governor mervyn king warned only last month that  ...

[1874_0] (business) score=0.376
bank voted 8-1 for no rate change the decision to keep interest rates on hold at 4.75% earlier this month was passed 8-1 by the bank of england s rate-setting body minutes have shown. one member of th ...

[1874_1] (business) score=0.359
in interest rates to 5%. he argued that economic growth was picking up and that the equity credit and housing markets had been stronger than expected. the bank s minutes said that risks to the inflati ...

[1088_3] (business) score=0.318
work to do in 2005 said avery shenfeld senior economist at cibc world markets. i think the federal reserve does not want to rock the boat and is using a gradual approach in raising the interest rate s ...


In [9]:
def show(query, k=3):
    print(f"\n=== {query} ===")
    for _, h in search_tfidf(query, k).iterrows():
        print(f"[{h['chunk_id']}] ({h['Category']}) score={h['score']:.3f}")
        print(h['chunk_text'][:150], "...")

show("cost of borrowing money")          # same idea as interest rates, different words
show("footballers signing new contracts")
show("computer viruses and online security")


=== cost of borrowing money ===
[1690_0] (politics) score=0.263
brown shrugs off economy fears gordon brown is to freeze petrol duty increases fund a £1bn package to avoid big council tax rises and boost childcare  ...
[192_1] (business) score=0.251
come. under direct threat will be the federal government s ability to make good on its debts to the social security trust fund. as soon as 2018 the fu ...
[334_0] (business) score=0.211
ecb holds rates amid growth fears the european central bank has left its key interest rate unchanged at 2% for the 19th month in succession. borrowing ...

=== footballers signing new contracts ===
[1127_2] (sport) score=0.153
senior appearances and scoring 25 goals before signing for sheffield united - where he scored 12 goals in 68 appearances. after signing for charlton h ...
[1831_4] (tech) score=0.151
and the first month under the new regime has seen many firms cautioned for not honouring them. some have been told to revamp websites so customers kno ..

In [10]:
!pip install -q sentence-transformers

In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(
    chunks_df['chunk_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(embeddings.shape)

def search_embed(query, k=5):
    q = model.encode([query], normalize_embeddings=True)
    scores = (embeddings @ q.T).ravel()
    top = scores.argsort()[::-1][:k]
    results = chunks_df.iloc[top].copy()
    results['score'] = scores[top]
    return results[['chunk_id', 'Category', 'score', 'chunk_text']]

def show_embed(query, k=3):
    print(f"\n=== {query} ===")
    for _, h in search_embed(query, k).iterrows():
        print(f"[{h['chunk_id']}] ({h['Category']}) score={h['score']:.3f}")
        print(h['chunk_text'][:150], "...")

show_embed("cost of borrowing money")
show_embed("footballers signing new contracts")
show_embed("computer viruses and online security")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/78 [00:00<?, ?it/s]

(4981, 384)

=== cost of borrowing money ===
[70_0] (business) score=0.481
indonesia declines debt freeze indonesia no longer needs the debt freeze offered by the paris club group of creditors economics minister aburizal bakr ...
[1690_4] (politics) score=0.479
and keep debt low and sustainable - both in this economic cycle and the next. borrowing figures for 2003/4 are £35bn - £2.5bn less than the £37.5bn pr ...
[248_2] (business) score=0.465
main topic of debate in the us presidential election. while demand for workers is far from booming the conditions are set for steady growth. overall c ...

=== footballers signing new contracts ===
[811_0] (sport) score=0.571
ronaldo considering new contract manchester united winger cristiano ronaldo said he is close to agreeing to a new contract at old trafford. the portug ...
[1588_0] (sport) score=0.523
wenger signs new deal arsenal manager arsene wenger has signed a new contract to stay at the club until may 2008. wenger has ended speculation

In [12]:
import numpy as np
np.save('bbc_embeddings.npy', embeddings)

from google.colab import files
files.download('bbc_embeddings.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 41.1 MB/s eta 0:00:00


In [14]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1])   # inner product = cosine for normalized vectors
index.add(embeddings.astype('float32'))
print("Vectors in index:", index.ntotal)

def search_faiss(query, k=5):
    q = model.encode([query], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(q, k)
    results = chunks_df.iloc[ids[0]].copy()
    results['score'] = scores[0]
    return results[['chunk_id', 'Category', 'score', 'chunk_text']]

print(search_faiss("footballers signing new contracts", 3)[['chunk_id', 'Category', 'score']])

Vectors in index: 4981
     chunk_id Category     score
1213    811_0    sport  0.570594
50     1588_0    sport  0.523034
4802   1525_0    sport  0.463713


In [15]:
!pip install -q transformers accelerate

In [16]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
lm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate(prompt, max_new_tokens=100):
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    out = lm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [17]:
def read_chunk(cid, n=600):
    text = chunks_df.loc[chunks_df['chunk_id'] == cid, 'chunk_text'].iloc[0]
    print(f"--- {cid} ---\n{text[:n]}\n")

for cid in ['1057_1', '1057_2', '1588_0']:
    read_chunk(cid)

for cid in ['90_0', '90_1', '1874_0']:
    read_chunk(cid)

--- 1057_1 ---
at old trafford but the game was followed by a now notorious food fight which saw ferguson s clothes covered in soup and pizza. the sides meet again at highbury on 1 february. in the tunnel wenger was criticising my players calling them cheats so i told him to leave them alone and behave himself ferguson said on saturday. he ran at me with hands raised saying what do you want to do about it to not apologise for the behaviour of the players to another manager is unthinkable. it s a disgrace but i don t expect wenger to ever apologise he s that type of person. those allegations were put to wenge

--- 1057_2 ---
which arsenal lost to slip 10 points behind chelsea in the title race. at first he said only: i ve always been consistent with that story and told you nothing happened. if he has to talk he talks. if he wants to make a newspaper article he makes a newspaper article. he doesn t interest me and doesn t matter to me at all. i will never answer to any provocation from h

In [18]:
test_queries = {
    "business": ["Company fraud of millions of dollars", "Germeny's economic decline", "Positive or negative economies and the effect on families"],
    "sport": ["most unexpected sport come backs", "soccer players leaves his team", "recoveries from injuries from sports"],
    "tech": ["how to improve sales of gadgets", "motion control gaming controls", "internet speed in mb"],
    "politics": ["Corruption of uk keeping money from europe budget", "Iraq war when soldsmith wasnt advised", "noward truent snoker hall"],
    "entertainment": ["film director becomes officer", "meet the forckers film success in money", "channel audiences and what they watch more"],
}

def precision_at_k(search_fn, k=5):
    per_cat = {}
    for cat, queries in test_queries.items():
        scores = [(search_fn(q, k)['Category'] == cat).mean() for q in queries]
        per_cat[cat] = sum(scores) / len(scores)
    return per_cat, sum(per_cat.values()) / len(per_cat)

for name, fn in [("TF-IDF", search_tfidf), ("Embeddings (FAISS)", search_faiss)]:
    per_cat, overall = precision_at_k(fn)
    print(f"{name}: overall precision@5 = {overall:.3f}")
    for c, v in per_cat.items():
        print(f"   {c:14s} {v:.3f}")

TF-IDF: overall precision@5 = 0.627
   business       0.667
   sport          0.533
   tech           0.733
   politics       0.467
   entertainment  0.733
Embeddings (FAISS): overall precision@5 = 0.840
   business       0.800
   sport          1.000
   tech           0.933
   politics       0.667
   entertainment  0.800


In [19]:
for q in test_queries["politics"]:
    print(f"\n=== {q!r} ===")
    print("TF-IDF:")
    for _, h in search_tfidf(q, 5).iterrows():
        print(f"  [{h['chunk_id']}] ({h['Category']}) {h['chunk_text'][:80]}...")
    print("Embeddings:")
    for _, h in search_faiss(q, 5).iterrows():
        print(f"  [{h['chunk_id']}] ({h['Category']}) {h['chunk_text'][:80]}...")


=== 'Corruption of uk keeping money from europe budget' ===
TF-IDF:
  [1342_1] (business) was rampant during the shevardnadze era and boost georgia s fragile public finan...
  [1791_0] (politics) brown names 16 march for budget chancellor gordon brown will deliver his budget ...
  [1527_1] (politics) would be a political price to pay for the uk s stance - britain contributed more...
  [1534_2] (politics) to come to an end. mr straw said that as well as the veto over the rebate the uk...
  [192_0] (business) the ticking budget facing the us the budget proposals laid out by the administra...
Embeddings:
  [778_1] (politics) the court has now qualified its opinion on the community accounts for a decade i...
  [778_0] (politics) eu fraud clampdown urged eu member states are failing to report fraud and irregu...
  [778_2] (politics) eu presidency. any fraud in other member states is potentially fraud against the...
  [1534_0] (politics) uk rebate unjustified - chirac french president jacqu

In [20]:
!pip install -q pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 13.4 MB/s eta 0:00:00


In [21]:
from spellchecker import SpellChecker

spell = SpellChecker()

def clean_query(query):
    words = query.lower().split()
    corrected = []
    for w in words:
        fix = spell.correction(w)
        corrected.append(fix if fix else w)
    return " ".join(corrected)

# Test it on your messy ones
for q in ["noward truent snoker hall", "wenegr arsnal contrct", "soldsmith wasnt advised"]:
    print(f"{q!r} -> {clean_query(q)!r}")

'noward truent snoker hall' -> 'toward truest smoker hall'
'wenegr arsnal contrct' -> 'wiener arsenal contact'
'soldsmith wasnt advised' -> 'goldsmith want advised'


In [22]:
messy = "noward truent snoker hall"
fixed = clean_query(messy)
print("Fixed:", fixed)

print("\nTF-IDF before:", search_tfidf(messy, 5)['Category'].tolist())
print("TF-IDF after: ", search_tfidf(fixed, 5)['Category'].tolist())
print("\nEmbeddings before:", search_faiss(messy, 5)['Category'].tolist())
print("Embeddings after: ", search_faiss(fixed, 5)['Category'].tolist())

Fixed: toward truest smoker hall

TF-IDF before: ['politics', 'entertainment', 'tech', 'entertainment', 'entertainment']
TF-IDF after:  ['politics', 'entertainment', 'tech', 'entertainment', 'entertainment']

Embeddings before: ['sport', 'sport', 'sport', 'sport', 'sport']
Embeddings after:  ['entertainment', 'sport', 'entertainment', 'sport', 'sport']


In [23]:
from spellchecker import SpellChecker

# Build a dictionary from your own corpus, not generic English
corpus_words = " ".join(chunks_df['chunk_text'].tolist()).split()
spell = SpellChecker()
spell.word_frequency.load_words(corpus_words)  # teach it corpus-specific words

def clean_query_v2(query):
    words = query.lower().split()
    corrected = [spell.correction(w) or w for w in words]
    return " ".join(corrected)

fixed_v2 = clean_query_v2(messy)
print("Corrected (corpus-aware):", fixed_v2)

print("\nTF-IDF after v2:", search_tfidf(fixed_v2, 5)['chunk_id'].tolist())
print("Embeddings after v2:", search_faiss(fixed_v2, 5)['chunk_id'].tolist())

Corrected (corpus-aware): toward truest smoker hall

TF-IDF after v2: ['1582_0', '1167_0', '28_1', '2029_1', '714_2']
Embeddings after v2: ['714_6', '485_1', '714_5', '2173_4', '559_0']


In [24]:
for messy in ["noward truent snoker hall", "wenegr arsnal contrct", "soldsmith wasnt advised"]:
    fixed = clean_query_v2(messy)
    print(f"{messy!r} -> {fixed!r}")

'noward truent snoker hall' -> 'toward truest smoker hall'
'wenegr arsnal contrct' -> 'wenger arsenal contact'
'soldsmith wasnt advised' -> 'goldsmith want advised'


In [25]:
# Embed whole articles (not chunks) for clustering — one vector per article
article_embeddings = model.encode(
    df['clean_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(article_embeddings.shape)

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

(1440, 384)


In [26]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(article_embeddings)

print(df['cluster'].value_counts())

cluster
4    344
3    327
1    262
0    260
2    247
Name: count, dtype: int64


In [27]:
import pandas as pd

comparison = pd.crosstab(df['cluster'], df['Category'])
print(comparison)

Category  business  entertainment  politics  sport  tech
cluster                                                 
0                1            249         0      3     7
1                7              2       251      1     1
2               11             11         3      0   222
3              315              1        10      0     1
4                1              0         2    338     3


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

def label_clusters(df, text_col='clean_text', cluster_col='cluster', top_n=8):
    vec = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=2)
    tfidf = vec.fit_transform(df[text_col])
    terms = vec.get_feature_names_out()

    for c in sorted(df[cluster_col].unique()):
        mask = (df[cluster_col] == c).values
        mean_tfidf = tfidf[mask].mean(axis=0).A1
        top_idx = mean_tfidf.argsort()[::-1][:top_n]
        top_words = [terms[i] for i in top_idx]
        dominant_cat = df.loc[mask, 'Category'].mode()[0]
        print(f"Cluster {c} (true majority: {dominant_cat}): {', '.join(top_words)}")

label_clusters(df)

Cluster 0 (true majority: entertainment): film, best, awards, music, band, award, actor, star
Cluster 1 (true majority: politics): mr, labour, election, blair, party, government, brown, minister
Cluster 2 (true majority: tech): people, mobile, software, users, phone, technology, net, music
Cluster 3 (true majority: business): growth, economy, firm, market, year, bank, company, sales
Cluster 4 (true majority: sport): england, game, win, cup, team, match, players, chelsea


In [29]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_query(query):
    return " ".join(lemmatizer.lemmatize(w) for w in query.lower().split())

# Quick test
for q in ["footballers signing new contracts", "players signed contracts", "companies raising prices"]:
    print(f"{q!r} -> {lemmatize_query(q)!r}")

'footballers signing new contracts' -> 'footballer signing new contract'
'players signed contracts' -> 'player signed contract'
'companies raising prices' -> 'company raising price'


In [30]:
from spellchecker import SpellChecker

corpus_words = " ".join(chunks_df['chunk_text'].tolist()).split()
spell_v2 = SpellChecker()
spell_v2.word_frequency.load_words(corpus_words)

In [31]:
def clean_query_full(query):
    """Spellcheck (corpus-aware) + lemmatize a raw user query."""
    corrected = " ".join(spell_v2.correction(w) or w for w in query.lower().split())
    return " ".join(lemmatizer.lemmatize(w) for w in corrected.split())

def search(query, method="embeddings", k=5, clean=True):
    """Search the corpus. method: 'tfidf' or 'embeddings'."""
    q = clean_query_full(query) if clean else query
    if method == "tfidf":
        return search_tfidf(q, k)
    elif method == "embeddings":
        return search_faiss(q, k)
    else:
        raise ValueError("method must be 'tfidf' or 'embeddings'")

def answer(question, k=3, clean=True):
    """Full RAG: clean query -> retrieve -> generate grounded answer with citations."""
    hits = search(question, method="embeddings", k=k, clean=clean)

    chunks = []
    for _, h in hits.iterrows():
        article_id = h['chunk_id'].split('_')[0]
        chunks.append({
            "text": h['chunk_text'],
            "source": f"ArticleId {article_id} ({h['Category']}), chunk {h['chunk_id']}",
        })

    context = "\n\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(chunks))
    prompt = (
        "Answer the question using ONLY the numbered passages below. "
        "If the passages don't contain the answer, say you don't know.\n\n"
        f"Passages:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    generated = generate(prompt)

    return {
        "question": question,
        "answer": generated,
        "sources": [c["source"] for c in chunks],
    }



In [32]:
result = answer("What did Arsene Wenger agree to?")
print(result["question"])
print(result["answer"])
print(result["sources"])

print("\n", search("footballers signing new contracts", method="tfidf", k=3)[['chunk_id','Category']])


What did Arsene Wenger agree to?
a long-term contract
['ArticleId 1588 (sport), chunk 1588_0', 'ArticleId 1057 (sport), chunk 1057_2', 'ArticleId 1588 (sport), chunk 1588_1']

      chunk_id Category
50     1588_0    sport
4652    565_0    sport
1213    811_0    sport


In [33]:
# Build once: article-level TF-IDF, kept separate from the chunk-level one used for search
article_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.8, min_df=2)
article_tfidf = article_vectorizer.fit_transform(df['clean_text'])

def get_cluster_label(cluster_id, top_n=8):
    """Return top keywords for a cluster id (0-4)."""
    mask = (df['cluster'] == cluster_id).values
    mean_tfidf = article_tfidf[mask].mean(axis=0).A1
    terms = article_vectorizer.get_feature_names_out()
    top_idx = mean_tfidf.argsort()[::-1][:top_n]
    return [terms[i] for i in top_idx]

print(get_cluster_label(4))

['england', 'game', 'win', 'cup', 'team', 'match', 'players', 'chelsea']


In [34]:
labeled_queries = {
    "worldcom fraud trial bernie ebbers": 1833,
    "european central bank leaves rate unchanged": 334,
    "indonesia turns down debt freeze offer": 70,
    "wenegr arsnal contrct": 1588,
    "will ronaldo stay at man utd": 811,
    "gerrard wants to stay at liverpool": 1525,
    "soldsmith wasnt advised": 1683,
    "noward truent snoker hall": 1582,
    "is the uk eu rebate up for negotiation": 1527,
    "microsoft tools to clean viruses and spyware": 1439,
    "hi-tech thieves and malicious programs": 1186,
    "playstation games based on harry potter": 2205,
    "french director alan parker gets honoured by france for his work in film, what award did he receive and why was he chosen?": 1797,
    "did meet the fockers really break a christmas day box office record in north america or was it just doing better than other films at the time": 1866,
    "what happened with snoop dogg and the make up artist who sued him for rape, and how much money was she asking": 1407,
}

def evaluate_recall(query_dict, method="embeddings", k=5, clean=True):
    hits_at_k = 0
    for q, correct_id in query_dict.items():
        results = search(q, method=method, k=k, clean=clean)
        found_ids = [int(cid.split('_')[0]) for cid in results['chunk_id']]
        hit = correct_id in found_ids
        hits_at_k += hit
        print(f"{'✓' if hit else '✗'} {q!r} -> looking for {correct_id}, got {found_ids}")
    print(f"\nRecall@{k}: {hits_at_k}/{len(query_dict)} = {hits_at_k/len(query_dict):.2f}")

print("=== TF-IDF ===")
evaluate_recall(labeled_queries, method="tfidf")
print("\n=== Embeddings ===")
evaluate_recall(labeled_queries, method="embeddings")

=== TF-IDF ===
✓ 'worldcom fraud trial bernie ebbers' -> looking for 1833, got [1413, 1238, 1143, 1833, 758]
✓ 'european central bank leaves rate unchanged' -> looking for 334, got [334, 154, 1874, 323, 173]
✓ 'indonesia turns down debt freeze offer' -> looking for 70, got [70, 1231, 70, 500, 575]
✓ 'wenegr arsnal contrct' -> looking for 1588, got [1588, 2061, 1559, 1487, 1487]
✓ 'will ronaldo stay at man utd' -> looking for 811, got [484, 811, 873, 1292, 873]
✓ 'gerrard wants to stay at liverpool' -> looking for 1525, got [1525, 950, 1887, 1525, 950]
✓ 'soldsmith wasnt advised' -> looking for 1683, got [310, 1683, 2170, 184, 310]
✓ 'noward truent snoker hall' -> looking for 1582, got [1582, 1167, 28, 2029, 714]
✓ 'is the uk eu rebate up for negotiation' -> looking for 1527, got [1527, 1534, 1534, 1534, 1534]
✓ 'microsoft tools to clean viruses and spyware' -> looking for 1439, got [1439, 1615, 809, 1439, 809]
✓ 'hi-tech thieves and malicious programs' -> looking for 1186, got [1186, 1

In [35]:
def matched_terms(query, chunk_text):
    q_vec = vectorizer.transform([query])
    feature_names = vectorizer.get_feature_names_out()
    q_terms = [feature_names[i] for i in q_vec.nonzero()[1]]
    text_words = set(chunk_text.lower().split())
    return [t for t in q_terms if t in text_words]

def most_similar_sentence(query, chunk_text):
    sentences = [s.strip() for s in chunk_text.split('. ') if s.strip()]
    if not sentences:
        return chunk_text[:100]
    sent_emb = model.encode(sentences, normalize_embeddings=True)
    q_emb = model.encode([query], normalize_embeddings=True)
    scores = (sent_emb @ q_emb.T).ravel()
    return sentences[scores.argmax()]

def search_explained(query, method="embeddings", k=5, clean=True):
    results = search(query, method=method, k=k, clean=clean)
    q = clean_query_full(query) if clean else query
    explanations = []
    for _, row in results.iterrows():
        if method == "tfidf":
            terms = matched_terms(q, row['chunk_text'])
            explanations.append(f"matched terms: {', '.join(terms)}" if terms else "no shared terms (weak match)")
        else:
            best = most_similar_sentence(q, row['chunk_text'])
            explanations.append(f"most similar sentence: \"{best[:120]}\"")
    results = results.copy()
    results['why'] = explanations
    return results[['chunk_id', 'Category', 'score', 'why']]

for method in ["tfidf", "embeddings"]:
    print(f"=== {method}, explained ===")
    for _, r in search_explained("wenegr arsnal contrct", method=method, k=3).iterrows():
        print(f"[{r['chunk_id']}] ({r['Category']}) score={r['score']:.3f}")
        print(f"  why: {r['why']}\n")

=== tfidf, explained ===
[1588_0] (sport) score=0.448
  why: matched terms: arsenal, wenger

[2061_0] (sport) score=0.430
  why: matched terms: arsenal, wenger

[1559_0] (sport) score=0.403
  why: matched terms: arsenal, wenger

=== embeddings, explained ===
[1057_2] (sport) score=0.644
  why: most similar sentence: "i have no diplomatic relations with him the arsenal boss is quoted as saying"

[1588_0] (sport) score=0.637
  why: most similar sentence: "wenger signs new deal arsenal manager arsene wenger has signed a new contract to stay at the club until may 2008"

[1327_2] (sport) score=0.627
  why: most similar sentence: "with arsenal hit by injuries and suspension inexperienced midfield pair of mathieu flamini and cesc fabregas will line u"



In [36]:
def demo(query, k=3):
    print("Raw query:    ", query)
    print("Cleaned query:", clean_query_full(query))
    for method in ["tfidf", "embeddings"]:
        print(f"\n--- {method} ---")
        for _, r in search_explained(query, method=method, k=k).iterrows():
            print(f"[{r['chunk_id']}] ({r['Category']}) score={r['score']:.3f}")
            print(f"  why: {r['why']}")

demo("wenegr arsnal contrct")

Raw query:     wenegr arsnal contrct
Cleaned query: wenger arsenal contact

--- tfidf ---
[1588_0] (sport) score=0.448
  why: matched terms: arsenal, wenger
[2061_0] (sport) score=0.430
  why: matched terms: arsenal, wenger
[1559_0] (sport) score=0.403
  why: matched terms: arsenal, wenger

--- embeddings ---
[1057_2] (sport) score=0.644
  why: most similar sentence: "i have no diplomatic relations with him the arsenal boss is quoted as saying"
[1588_0] (sport) score=0.637
  why: most similar sentence: "wenger signs new deal arsenal manager arsene wenger has signed a new contract to stay at the club until may 2008"
[1327_2] (sport) score=0.627
  why: most similar sentence: "with arsenal hit by injuries and suspension inexperienced midfield pair of mathieu flamini and cesc fabregas will line u"


In [ ]:
!pip install -q gradio
import gradio as gr

text_lookup = dict(zip(chunks_df['chunk_id'], chunks_df['chunk_text']))

css = """
.gradio-container {
    background: linear-gradient(135deg, #eef2ff 0%, #faf5ff 50%, #fff1f2 100%) !important;
    font-family: 'Inter', system-ui, sans-serif !important;
}
footer { display: none !important; }
.hero { text-align: center; padding: 28px 10px 6px; }
.hero h1 {
    font-size: 2.6rem; margin: 0; font-weight: 800;
    background: linear-gradient(90deg, #4f46e5, #9333ea, #ec4899);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
}
.hero p { color: #475569; margin-top: 8px; font-size: 1.05rem; }
.card {
    background: #ffffff !important; border-radius: 16px !important;
    border: 1px solid #e0e7ff !important; padding: 18px !important;
    box-shadow: 0 6px 24px rgba(79, 70, 229, 0.08) !important;
}
.go-btn {
    background: linear-gradient(90deg, #4f46e5, #9333ea) !important;
    color: #ffffff !important; border: none !important;
    border-radius: 12px !important; font-weight: 600 !important;
}
.go-btn:hover { filter: brightness(1.08); }
"""
theme = gr.themes.Soft(primary_hue="indigo", secondary_hue="purple", neutral_hue="slate")

def ui_search(query, method, k):
    if not query.strip():
        return "", []
    res = search_explained(query, method=method, k=int(k))
    rows = [[r['chunk_id'], r['Category'], round(float(r['score']), 3),
             text_lookup[r['chunk_id']][:180] + "...", r['why']]
            for _, r in res.iterrows()]
    return clean_query_full(query), rows

def ui_answer(question):
    if not question.strip():
        return "", ""
    r = answer(question)
    return r["answer"], "\n".join(r["sources"])

try:
    app = gr.Blocks(title="DocuSense", theme=theme, css=css)
except TypeError:
    app = gr.Blocks(title="DocuSense")

with app:
    gr.HTML(f"""<style>{css}</style>
    <div class='hero'>
      <h1>DocuSense</h1>
      <p>Search 1,440 BBC News articles</p>
    </div>""")

    with gr.Tab("Search"):
        with gr.Column(elem_classes="card"):
            with gr.Row():
                q = gr.Textbox(label="What are you looking for?", scale=4,
                               placeholder="e.g. wenegr arsnal contrct")
                btn = gr.Button("Search", elem_classes="go-btn", scale=1)
            with gr.Row():
                method = gr.Radio(["embeddings", "tfidf"], value="embeddings", label="Retriever")
                k = gr.Slider(1, 10, value=5, step=1, label="Number of results")
            gr.Examples(
                examples=[["wenegr arsnal contrct"], ["who is suing snoop dogg"],
                          ["microsoft tools to remove spyware"], ["bank of england interest rates"]],
                inputs=[q], label="Try one of these")
        with gr.Column(elem_classes="card"):
            cleaned = gr.Textbox(label="Query after cleaning", interactive=False)
            table = gr.Dataframe(
                headers=["chunk", "category", "score", "passage", "why it ranked"],
                wrap=True, interactive=False)
        btn.click(ui_search, [q, method, k], [cleaned, table])
        q.submit(ui_search, [q, method, k], [cleaned, table])

    with gr.Tab("Ask a question"):
        with gr.Column(elem_classes="card"):
            with gr.Row():
                question = gr.Textbox(label="Ask about the articles", scale=4,
                                      placeholder="e.g. What did Arsene Wenger agree to?")
                ask = gr.Button("Answer", elem_classes="go-btn", scale=1)
            gr.Examples(
                examples=[["What did Arsene Wenger agree to?"],
                          ["What is Microsoft releasing to deal with viruses?"]],
                inputs=[question], label="Try one of these")
        with gr.Column(elem_classes="card"):
            ans = gr.Textbox(label="Answer", interactive=False)
            src = gr.Textbox(label="Sources", interactive=False)
        ask.click(ui_answer, [question], [ans, src])
        question.submit(ui_answer, [question], [ans, src])

try:
    app.launch(share=True, theme=theme, css=css)
except TypeError:
    app.launch(share=True)